# Predicting Aircraft Fair Market Value — Extended Project
## Building an Independent Appraisal Model for Aircraft Trading

**Skeleton Notebook** — instructions and structure only. Fill in every `# TODO` cell yourself.

### Project brief
You're a data scientist at an aircraft leasing and trading desk. Your job is to build an **independent appraisal model** that estimates an aircraft's fair market value purely from its technical condition and utilization history — the same kind of work an ISTAT-certified aircraft appraiser does professionally. If your model's estimate disagrees meaningfully with a broker's asking price, that's a candidate deal worth flagging for deeper due diligence. You're given `aircraft_valuation.csv` (1,350 aircraft records, 24 raw columns).

This extended project mirrors a real asset-valuation workflow: a data-quality audit (including a classic pandas gotcha), unit-stripping across half a dozen aerospace-specific measurement conventions, ordinal encoding of maintenance-condition categories, a **leakage audit** for market-quote-like columns, a comparison across linear, log-linear, and tree-ensemble models, hyperparameter tuning, feature importance, and a backtest of a simple "find underpriced aircraft" strategy against a held-out comparable-sale price.

> **This is a synthetic dataset built for a machine-learning exercise.** The aircraft records, valuations, and comparable sales are simulated, not real transaction data. Nothing in this notebook is investment or appraisal advice — the backtest section demonstrates a methodology, not a validated trading strategy.

### How to use this notebook
Each section has a short **Context** explaining *why* the step matters, then a **Task** list of exactly what to build. Use the **Cheat Sheet** notebook for syntax help and the **Background Theory** notebook for conceptual grounding. Don't peek at the Solutions notebook until you've attempted each section yourself.

### Success criteria for the whole project
- A cleaned, fully numeric feature matrix with no leakage between train and test — **and no leakage from market-quote-style columns into your features**
- At least 4 trained regression models compared fairly, including at least one on a transformed target
- A tuned final model with cross-validated performance estimates
- Two independent feature-importance views that agree on the top value drivers
- A backtest comparing your model's disagreements with the broker's asking price against a held-out comparable-sale price


## Module 0 — Environment Setup

**Context.** Fixed seeds and consistent imports make your results comparable to the Solutions notebook.

**Task**
- Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`.
- Import `train_test_split`, `cross_val_score`, `RandomizedSearchCV` from `sklearn.model_selection`.
- Import `LinearRegression` from `sklearn.linear_model`.
- Import `RandomForestRegressor`, `GradientBoostingRegressor` from `sklearn.ensemble`.
- Import `mean_absolute_error`, `mean_squared_error`, `r2_score` from `sklearn.metrics`.
- Set `RANDOM_STATE = 42` and use it everywhere a `random_state` argument exists.


In [ ]:
# TODO: imports and global constants


## Module 1 — Data Loading & Initial Inspection

**Context.** Aircraft records mix identifiers, condition categories, and unit-bearing measurements from several different aerospace conventions (flight hours, cycles, kg/hr fuel burn).

**Task**
- Load `aircraft_valuation.csv` into `df`.
- Print `df.shape`, `df.head()`, `df.info()`.
- Print `df.describe(include='all').T`.
- List which columns are numeric-looking but stored as text (hint: look for `"hrs"`, `"cycles"`, `"kg/hr"`, `"seats"`, `"ADs"`).


In [ ]:
# TODO: load data and inspect


## Module 2 — Data Quality Audit (including a classic pandas gotcha)

**Context.** This dataset has a trap that catches even experienced analysts: `pandas.read_csv` treats several literal strings — including the word `"None"` — as missing values *by default*. `damage_history` genuinely has a category called `"None"` (meaning "no damage history"), but if you only look at `.isnull().sum()` without checking *why* those rows are null, you might wrongly conclude the column has a data-collection gap instead of a real category being silently swallowed by pandas' defaults.

**Task**
- Build a dtype/nunique/missing audit table.
- Check `df.duplicated().sum()`; inspect flagged rows before dropping.
- Look specifically at `damage_history`: how many nulls does `.isnull().sum()` report? Then check `pd.read_csv`'s documentation (or just reason about it) for why the string `"None"` might have been swallowed. Confirm your hypothesis and fix it — **without accidentally treating genuine missing data the same way you treat this recovered category.**
- Normalize `manufacturer` casing (it's scattered across multiple cases, like the previous projects' brand/sportsbook casing issues).


In [ ]:
# TODO: dtype audit table


In [ ]:
# TODO: duplicate check


In [ ]:
# TODO: investigate and fix the damage_history 'None'-as-NaN issue


In [ ]:
# TODO: normalize manufacturer casing


## Module 3 — Feature Engineering I: Unit-Bearing Measurements

**Context.** `total_flight_hours`, `total_cycles`, `engine_hours_since_overhaul`, and `engine_cycles_since_overhaul` all carry thousands-commas plus a unit suffix. `seat_count` mixes a numeric seat count with a special `"Freighter (0 seats)"` case. `fuel_burn_kg_per_hr` needs the same comma+unit treatment.

**Task**
- Strip the thousands-comma and unit suffix (`" hrs"` or `" cycles"`) from all four hour/cycle columns; cast to `float`.
- Extract the numeric seat count from `seat_count` (regex is your friend here) and create an `is_freighter` flag for the zero-seat case.
- Clean `fuel_burn_kg_per_hr` the same way (comma + `" kg/hr"`).


In [ ]:
# TODO: clean hour/cycle columns


In [ ]:
# TODO: parse seat_count + is_freighter flag


In [ ]:
# TODO: clean fuel_burn_kg_per_hr


## Module 4 — Feature Engineering II: Disguised Missing Values & Yes/No Flags

**Context.** `airworthiness_directives_open` mixes a count (`"2 ADs"`) with a `"Not Reported"` sentinel — the same disguised-missing pattern you've seen in prior projects, requiring both an imputed numeric value and a missingness flag.

**Task**
- Clean `airworthiness_directives_open`: strip `" ADs"`, replace `"Not Reported"` with a sentinel, cast to `int`, create an `ad_count_unreported` flag, and impute the sentinel rows with the column's median (excluding the sentinel).
- Convert `winglets_installed` and `avionics_upgrade` from `"Yes"/"No"` to `0/1`.


In [ ]:
# TODO: airworthiness_directives_open cleaning + missing flag


In [ ]:
# TODO: Yes/No flag conversions


## Module 5 — Feature Engineering III: Ordinal Encoding & Domain-Derived Features

**Context.** `airframe_check_status`, `damage_history`, and `paint_condition` aren't just categories — they have a natural **order** (a fresher D-check is unambiguously better than a check that's due; no damage is better than minor damage, which is better than major damage). Treating them as unordered one-hot categories would throw away that ordering information.

**Task**
- Map `airframe_check_status` to an ordinal score (e.g. `D-Check Due`=0, `C-Check Due`=1, `Fresh C-Check`=2, `Fresh D-Check`=3).
- Map `damage_history` to an ordinal score (`Major`=0, `Minor`=1, `None`=2).
- Map `paint_condition` to an ordinal score (`Poor`=0, `Fair`=1, `Good`=2, `Excellent`=3).
- Create `age_years` from `manufacture_year` (use 2025 as the current year).
- Create `engine_overhaul_pct_remaining` — a normalized measure of how much life remains before the next engine overhaul is due (hint: assume a ~9,000-hour overhaul interval).
- Justify in one sentence why ordinal encoding is more appropriate than one-hot encoding for these three columns specifically.


In [ ]:
# TODO: ordinal-encode airframe_check_status, damage_history, paint_condition


In [ ]:
# TODO: age_years, engine_overhaul_pct_remaining


## Module 6 — Advanced EDA & the Market-Quote Leakage Audit

**Context.** Just like the sports-betting project's opening spread and moneyline, this dataset has two columns that are dangerously close to the target: `broker_asking_price_usd` and `recent_comparable_sale_price_usd`. Both are ultimately anchored to the same underlying "true value" as `appraised_fair_market_value_usd` — using either as a feature would let the model shortcut to "read off a number that already encodes the answer" instead of demonstrating that technical/condition data alone can approximate a fair value.

**Task**
- Plot the distribution of `appraised_fair_market_value_usd`; compute its skewness. Given what you know about how assets depreciate (a *percentage* per year, not a flat dollar amount per year), consider whether a log transform of the target might make later modeling easier.
- Build a correlation heatmap **including** `broker_asking_price_usd` and `recent_comparable_sale_price_usd`. Compute their exact correlations with the target.
- Write one paragraph on which of the two columns should be excluded from your feature set, and why one might be closer to "legitimate market context" and the other closer to "outright leakage" (hint: consider timing — which of the two is more like a rumor/quote, and which is more like a nearly-settled fact about the same value you're trying to estimate?).
- Compute VIF on your remaining numeric features.
- Boxplot the target by `lease_status` and by `damage_history`.
- Run an IQR-based outlier check; given the wide range of aircraft types (regional jets to widebodies), think carefully about whether IQR-based outlier detection even makes sense on the raw scale here, or whether it should be applied after a transform.


In [ ]:
# TODO: target distribution + skew; consider a log transform


In [ ]:
# TODO: correlation heatmap INCLUDING the market-quote columns


In [ ]:
# TODO: explicit correlation check + written leakage decision


In [ ]:
# TODO: VIF check


In [ ]:
# TODO: boxplots + outlier check (with a note on IQR validity across very different aircraft sizes)


## Module 7 — Encoding Remaining Categorical Variables (Leakage-Safe)

**Context.** `aircraft_type`, `manufacturer`, `engine_type`, and `operator_region` are nominal (unordered) categoricals of varying cardinality; `maintenance_program` and `lease_status` are low-cardinality nominal categoricals.

**Task**
- Using the `< 5 unique values` threshold, split the remaining (non-ordinal) categorical columns into one-hot vs. target-encode groups.
- Split into train/test **before** computing any target-encoding statistic.
- Fit target-encoding means on `y_train`/`X_train` only; apply to `X_test` with an unseen-category fallback to the training global mean.
- Confirm zero `NaN`s remain in `X_train`/`X_test`.


In [ ]:
# TODO: bucket remaining categoricals by cardinality


In [ ]:
# TODO: train/test split, then fit + apply leakage-safe target encoding


## Module 8 — Baseline, Linear, and Log-Linear Models

**Context.** Asset depreciation is typically a *multiplicative* process (an aircraft loses roughly a percentage of its remaining value each year, not a fixed dollar amount) — which means a plain linear model on the raw-dollar target may underperform a linear model fit to the **log** of the target, since a multiplicative process becomes additive after a log transform.

**Task**
- Build a mean-predictor baseline; report MAE/RMSE/R² on the raw-dollar test target.
- Fit `LinearRegression` on the raw-dollar target; report the same three metrics.
- Fit a second `LinearRegression` on `log(y_train)`, then exponentiate its predictions back to dollars (`np.exp(...)`) before computing MAE/RMSE/R² — so every model is compared on the same dollar scale.
- Compare the two linear approaches. Which fits better, and does that match your Module 6 hypothesis about a multiplicative process?


In [ ]:
# TODO: mean-predictor baseline


In [ ]:
# TODO: LinearRegression on raw target


In [ ]:
# TODO: LinearRegression on log(target), exponentiate predictions back to dollars


## Module 9 — Tree-Ensemble Models

**Task**
- Train `RandomForestRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- Train `GradientBoostingRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- Collect every model trained so far into one comparison table sorted by MAE.


In [ ]:
# TODO: RandomForestRegressor, GradientBoostingRegressor


In [ ]:
# TODO: model comparison table


## Module 10 — Hyperparameter Tuning & Cross-Validation

**Task**
- Run 5-fold `cross_val_score` (scoring `'neg_mean_absolute_error'`) on your best model from Module 9.
- Define a hyperparameter search space and run `RandomizedSearchCV` (`cv=5`).
- Report best params, best CV score, and the refit model's test-set performance.


In [ ]:
# TODO: cross_val_score on your leading candidate


In [ ]:
# TODO: RandomizedSearchCV, refit, evaluate on test set


## Module 11 — Evaluation & Diagnostics

**Task**
- Report MAE, RMSE, R², and MAPE for your final model (MAPE should behave reasonably here since aircraft values never approach zero — unlike the sports-betting project's near-zero spread problem).
- Plot predicted vs. actual value with a `y=x` reference line.
- Plot residuals vs. predicted values; check whether error variance grows with aircraft value (a common pattern for wide-ranging dollar targets) and consider whether this supports modeling in log-space.


In [ ]:
# TODO: MAE, RMSE, R2, MAPE


In [ ]:
# TODO: predicted vs actual scatterplot + residual plots


## Module 12 — Feature Importance

**Task**
- Plot the top 10 features by impurity-based importance (tree model) or absolute coefficient (linear/log-linear model).
- Compute and plot permutation importance on the test set.
- Compare the two — do `age_years`, `check_status_score`, `damage_score`, and aircraft type/manufacturer dominate, as domain knowledge would predict?


In [ ]:
# TODO: importance plot #1


In [ ]:
# TODO: permutation importance plot


## Module 13 — Backtest: Finding Underpriced Aircraft

**Context.** If your model's fair-value estimate is meaningfully *higher* than a broker's asking price, that's a candidate "good deal." `recent_comparable_sale_price_usd` was excluded from your features the whole time — bring it back now, purely to check whether flagged deals were also validated by the (held-out) comparable-sale evidence.

> Remember: this is a synthetic dataset. This section demonstrates a backtesting methodology, not investment advice.

**Task**
- For your test-set aircraft, compute `edge = model_prediction - broker_asking_price_usd`.
- Flag aircraft where `edge` exceeds some threshold (e.g. 8% of the asking price) as "candidate underpriced deals."
- Using `recent_comparable_sale_price_usd` (never used as a training feature), check what fraction of flagged deals also had a comparable sale meaningfully above the asking price — a proxy for "the broader market agreed this aircraft was underpriced."
- Compare that confirmation rate against the same check applied to *all* test aircraft (not just flagged ones).
- Write two sentences interpreting the result, including at least one reason to be cautious about a good-looking backtest on a small/synthetic sample.


In [ ]:
# TODO: compute edge, flag candidate underpriced aircraft


In [ ]:
# TODO: confirmation-rate check against recent_comparable_sale_price_usd, vs baseline


## Module 14 — Persistence & Inference

**Task**
- Save your final model with `joblib.dump`, along with encoding maps, ordinal maps, and the training column order.
- Write a function `predict_fair_value(raw_aircraft_dict)` that takes a dictionary shaped like one row of the raw CSV (minus the market-quote columns) and returns your model's fair-value estimate in dollars.
- Test it on 2–3 made-up aircraft and sanity-check the outputs are plausible values for those aircraft types.


In [ ]:
# TODO: persist model + encoders


In [ ]:
# TODO: predict_fair_value(raw_aircraft_dict) function + sanity checks


## Module 15 — Conclusions & Write-Up

**Task**
- Write a 150–250 word summary covering: which model you chose and why (including whether the log transform mattered), its expected error margin, the top 3–5 value drivers, the backtest result and at least one reason for caution, and one concrete next step.


*(Write your conclusions here.)*